In [3]:
# %% [markdown]
# # Step 2: Portable U2Net Background Removal (Memory Optimized - No Display)
# ## Same training pipeline but with maximum memory efficiency and portability

# %%
import torch
import numpy as np
import cv2
from torchvision import transforms
from PIL import Image
import sys
import os
import gc
import urllib.request
import zipfile
import shutil
import warnings
warnings.filterwarnings('ignore')

# Force CPU and aggressive memory settings
torch.set_num_threads(1)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# %%
# Get current directory
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Create u2net folder structure
u2net_folder = os.path.join(current_dir, 'u2net')
model_folder = os.path.join(u2net_folder, 'model')
weights_folder = os.path.join(u2net_folder, 'weights')
os.makedirs(model_folder, exist_ok=True)
os.makedirs(weights_folder, exist_ok=True)
print(f"✅ Created U2Net folders")

# %% [markdown]
# ### Check if weights are downloaded

# %%
weights_path = os.path.join(weights_folder, 'u2net.pth')
if os.path.exists(weights_path):
    file_size = os.path.getsize(weights_path) / (1024 * 1024)  # MB
    print(f"✅ U2Net weights found: {weights_path}")
    print(f"   File size: {file_size:.2f} MB")
else:
    print(f"❌ U2Net weights not found at: {weights_path}")
    print("   Please download from: https://drive.google.com/uc?id=1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ")

# %% [markdown]
# ### Helper function to get a test image from the project

# %%
def get_test_image():
    """Get the first image from static/uploads folder or create a dummy image"""
    # First, check static/uploads folder for existing images
    upload_folder = os.path.join(current_dir, 'static', 'uploads')
    if os.path.exists(upload_folder):
        images = [f for f in os.listdir(upload_folder) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]
        if images:
            return os.path.join(upload_folder, images[0])
    
    # If no images found, check if there's a sample image in test_output
    test_output = os.path.join(current_dir, 'test_output')
    if os.path.exists(test_output):
        images = [f for f in os.listdir(test_output) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if images:
            return os.path.join(test_output, images[0])
    
    # If no images found, return None (will use dummy image)
    return None

# %% [markdown]
# ### Portable U2Net Background Remover Class

# %%
class BackgroundRemover:
    """
    Portable U2Net Background Remover
    Automatically downloads U2Net if not found
    Memory-optimized - no display, just processing
    """
    
    def __init__(self, model_path=None):
        self.device = torch.device('cpu')
        print(f"Using device: {self.device}")
        
        # Get paths
        self.current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
        self.project_root = os.path.dirname(self.current_dir) if '__file__' in dir() else self.current_dir
        
        # Set up paths
        self.u2net_dir = os.path.join(self.project_root, 'u2net')
        self.model_dir = os.path.join(self.u2net_dir, 'model')
        self.weights_dir = os.path.join(self.u2net_dir, 'weights')
        
        # Create directories
        os.makedirs(self.model_dir, exist_ok=True)
        os.makedirs(self.weights_dir, exist_ok=True)
        
        # Download/verify U2Net source
        self.u2net_available = self._ensure_u2net_source()
        
        # Set model path
        if model_path is None:
            model_path = os.path.join(self.weights_dir, 'u2net.pth')
        self.model_path = model_path
        
        # Import and load model
        self._import_u2net()
        self._load_model()
        
        # Training transform
        self.transform = transforms.Compose([
            transforms.Resize((320, 320)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
        ])
        
        gc.collect()
        if self.model is not None:
            print("✅ BackgroundRemover initialized with U2Net model")
        else:
            print("✅ BackgroundRemover initialized in fallback mode")
    
    def _ensure_u2net_source(self):
        """Download U2Net source if not present"""
        u2net_py = os.path.join(self.model_dir, 'u2net.py')
        
        if not os.path.exists(u2net_py):
            print("\n📥 Downloading U2Net source...")
            u2net_url = "https://github.com/xuebinqin/U-2-Net/archive/master.zip"
            zip_path = os.path.join(self.u2net_dir, 'u2net.zip')
            
            try:
                # Download
                urllib.request.urlretrieve(u2net_url, zip_path)
                print("  ✅ Downloaded")
                
                # Extract
                print("  📦 Extracting...")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(self.u2net_dir)
                
                # Move model files
                extracted = os.path.join(self.u2net_dir, 'U-2-Net-master')
                if os.path.exists(extracted):
                    src_model = os.path.join(extracted, 'model')
                    if os.path.exists(src_model):
                        for file in os.listdir(src_model):
                            if file.endswith('.py'):
                                shutil.copy(
                                    os.path.join(src_model, file),
                                    os.path.join(self.model_dir, file)
                                )
                        print("  ✅ Model files copied")
                
                # Clean up
                os.remove(zip_path)
                if os.path.exists(extracted):
                    shutil.rmtree(extracted)
                
                print("✅ U2Net source downloaded successfully")
                return True
                
            except Exception as e:
                print(f"⚠️ Could not download U2Net: {e}")
                print("   Please manually download from:")
                print("   https://github.com/xuebinqin/U-2-Net")
                print(f"   and place u2net.py in: {self.model_dir}")
                return False
        else:
            print("✅ U2Net source already present")
            return True
    
    def _import_u2net(self):
        """Import U2Net module"""
        if self.model_dir not in sys.path:
            sys.path.insert(0, self.model_dir)
        
        try:
            from u2net import U2NET
            self.U2NET = U2NET
            self.u2net_available = True
            print("✅ U2Net module imported")
        except ImportError as e:
            print(f"⚠️ Could not import U2Net: {e}")
            print("   Will use fallback mode")
            self.U2NET = None
            self.u2net_available = False
    
    def _load_model(self):
        """Load the U2Net model"""
        if self.U2NET is None:
            self.model = None
            print("⚠️ Running in fallback mode (U2Net module missing)")
            return
            
        if not os.path.exists(self.model_path):
            print(f"⚠️ Model weights not found at: {self.model_path}")
            print("   Will use fallback mode")
            self.model = None
            return
        
        try:
            print(f"📦 Loading U2Net model from: {self.model_path}")
            self.model = self.U2NET(in_ch=3, out_ch=1)
            state_dict = torch.load(
                self.model_path, 
                map_location=self.device,
                weights_only=True
            )
            self.model.load_state_dict(state_dict)
            self.model.to(self.device)
            self.model.eval()
            print("✅ U2Net model loaded successfully!")
            print(f"   Model type: {type(self.model).__name__}")
        except Exception as e:
            print(f"⚠️ Could not load model: {e}")
            self.model = None
    
    def remove_background(self, image, target_size=(224, 224), max_size=800):
        """
        Memory-optimized background removal - preserves original leaf colors
        
        Args:
            image: numpy array or PIL Image
            target_size: final size for model input
            max_size: maximum dimension for processing
            
        Returns:
            result: numpy array with background removed (original colors preserved)
            mask: binary mask from U2Net
        """
        # Fallback if model not loaded
        if self.model is None:
            if isinstance(image, np.ndarray):
                return cv2.resize(image, target_size), None
            elif isinstance(image, Image.Image):
                return np.array(image.resize(target_size)), None
            return image, None
        
        try:
            # Store original image for color preservation
            original_image = image.copy() if isinstance(image, np.ndarray) else image
            
            # Resize large images for processing
            if isinstance(image, np.ndarray):
                h, w = image.shape[:2]
                if max(h, w) > max_size:
                    scale = max_size / max(h, w)
                    new_w = int(w * scale)
                    new_h = int(h * scale)
                    image = cv2.resize(image, (new_w, new_h))
                
                # Convert to PIL for U2Net
                image_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            elif isinstance(image, Image.Image):
                if max(image.size) > max_size:
                    scale = max_size / max(image.size)
                    new_size = tuple(int(dim * scale) for dim in image.size)
                    image_pil = image.resize(new_size, Image.Resampling.LANCZOS)
                else:
                    image_pil = image
            
            # Get mask from U2Net
            input_tensor = self.transform(image_pil).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                d1, d2, d3, d4, d5, d6, d7 = self.model(input_tensor)
            
            mask = d1[:, 0, :, :].cpu().numpy()
            
            # Clean up
            del d1, d2, d3, d4, d5, d6, d7, input_tensor
            gc.collect()
            
            # Process mask
            mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
            mask = mask.transpose(1, 2, 0)
            
            # Resize mask to target size
            mask_target = cv2.resize(mask, target_size)
            mask_target = (mask_target * 255).astype(np.uint8)
            
            # === FIX: Preserve original colors ===
            # Get the original image resized to target size (without any color changes)
            if isinstance(original_image, np.ndarray):
                # Resize original image to target size
                image_resized = cv2.resize(original_image, target_size)
                # Ensure it's RGB
                if len(image_resized.shape) == 3 and image_resized.shape[2] == 3:
                    # Keep as is - it's already in correct color space
                    pass
            else:
                # It was a PIL Image
                image_resized = np.array(original_image.resize(target_size, Image.Resampling.LANCZOS))
            
            # Create result by copying the resized original
            result = image_resized.copy()
            
            # Apply mask - set background pixels to white, keep leaf pixels original
            # For RGB images, we need to handle 3-channel mask properly
            if len(mask_target.shape) == 3 and mask_target.shape[2] == 1:
                mask_target = mask_target.squeeze()
            
            # Set background (mask < threshold) to white
            result[mask_target < 128] = [255, 255, 255]
            
            # Debug info
            print(f"  🎨 Background removal - Original colors preserved")
            print(f"     Mask range: [{mask_target.min()}, {mask_target.max()}]")
            print(f"     Background pixels: {np.sum(mask_target < 128)}/{mask_target.size}")
            
            return result, mask_target
            
        except Exception as e:
            print(f"⚠️ Warning in background removal: {e}")
            import traceback
            traceback.print_exc()
            # Fallback: return resized original
            if isinstance(image, Image.Image):
                return np.array(image.resize(target_size)), None
            elif isinstance(image, np.ndarray):
                return cv2.resize(image, target_size), None
            return image, None
    
    def process_and_save(self, image_path, output_dir, target_size=(224, 224)):
        """
        Process an image and save results without displaying
        """
        try:
            # Read image
            img = cv2.imread(image_path)
            if img is None:
                print(f"❌ Could not read image: {image_path}")
                return None
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Remove background
            result, mask = self.remove_background(img_rgb, target_size=target_size)
            
            # Save results
            os.makedirs(output_dir, exist_ok=True)
            
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            
            # Save original resized
            orig_resized = cv2.resize(img_rgb, target_size)
            orig_resized_bgr = cv2.cvtColor(orig_resized, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_original.jpg"), orig_resized_bgr)
            
            # Save mask if available
            if mask is not None:
                cv2.imwrite(os.path.join(output_dir, f"{base_name}_mask.jpg"), mask)
            
            # Save result
            result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_bg_removed.jpg"), result_bgr)
            
            print(f"✅ Saved results to {output_dir}")
            
            return {
                'original': os.path.join(output_dir, f"{base_name}_original.jpg"),
                'mask': os.path.join(output_dir, f"{base_name}_mask.jpg") if mask is not None else None,
                'result': os.path.join(output_dir, f"{base_name}_bg_removed.jpg")
            }
            
        except Exception as e:
            print(f"❌ Error in process_and_save: {e}")
            return None
    
    def __del__(self):
        """Cleanup when object is deleted"""
        try:
            import gc
            gc.collect()
        except:
            pass

# %% [markdown]
# ### Test the Portable Background Remover with Actual Weights

# %%
print("\n" + "="*70)
print("🧪 TESTING PORTABLE BACKGROUND REMOVER WITH ACTUAL WEIGHTS")
print("="*70)

# Check if weights exist
if os.path.exists(weights_path):
    print(f"\n✅ Found weights at: {weights_path}")
    test_mode = "full"
else:
    print("\n⚠️ U2Net weights not found!")
    print("   Testing in fallback mode (no actual background removal)")
    test_mode = "fallback"

# Initialize remover with weights
print("\n🔄 Initializing BackgroundRemover...")
remover = BackgroundRemover(weights_path if os.path.exists(weights_path) else None)

# Get test image from project folders
test_img = get_test_image()

if test_img and os.path.exists(test_img):
    print(f"\n📷 Testing with: {os.path.basename(test_img)}")
    
    # Read image
    img = cv2.imread(test_img)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"  Original shape: {img_rgb.shape}")
    print(f"  Original dtype: {img_rgb.dtype}")
    print(f"  Original range: [{img_rgb.min()}, {img_rgb.max()}]")
    
    # Test 1: Basic removal
    print("\n🔄 Test 1: Basic background removal...")
    result, mask = remover.remove_background(img_rgb, target_size=(224, 224))
    
    print(f"  ✅ Result shape: {result.shape}")
    print(f"  ✅ Result dtype: {result.dtype}")
    print(f"  ✅ Result range: [{result.min()}, {result.max()}]")
    
    if mask is not None:
        print(f"  ✅ Mask shape: {mask.shape}")
        print(f"  ✅ Mask range: [{mask.min()}, {mask.max()}]")
    else:
        print(f"  ℹ️ No mask generated (fallback mode)")
    
    # Test 2: Process and save
    print("\n🔄 Test 2: Process and save...")
    output_dir = os.path.join(current_dir, 'test_output_with_weights')
    saved = remover.process_and_save(test_img, output_dir)
    
    if saved:
        print(f"  ✅ Files saved to: {output_dir}")
        for key, path in saved.items():
            if path:
                file_size = os.path.getsize(path) / 1024  # KB
                print(f"     - {key}: {os.path.basename(path)} ({file_size:.1f} KB)")
    
    print("\n✅ All tests completed!")
    print(f"   Mode: {test_mode.upper()}")
    print(f"\n📁 Check the saved images in: {output_dir}")
    
    # Clean up
    del remover
    gc.collect()
    
else:
    print(f"\n⚠️ No test image found in static/uploads/ or test_output/")
    print("   Creating dummy image for testing...")
    
    # Create dummy image
    dummy = np.zeros((500, 500, 3), dtype=np.uint8)
    # Make it a green square on light blue background
    dummy[:, :, 0] = 100  # Blue component
    dummy[:, :, 1] = 150  # Green component
    dummy[:, :, 2] = 200  # Red component
    # Draw a green square in the center (simulating leaf)
    cv2.rectangle(dummy, (100, 100), (400, 400), (0, 255, 0), -1)
    # Add some texture inside the square
    cv2.rectangle(dummy, (150, 150), (350, 350), (0, 200, 0), -1)
    
    result, mask = remover.remove_background(dummy)
    print(f"✅ Dummy test passed - Result shape: {result.shape}")
    if mask is not None:
        print(f"✅ Mask shape: {mask.shape}")
    else:
        print(f"ℹ️ No mask generated (fallback mode)")

# %% [markdown]
# ### Save the portable class for web app

# %%
# Save the portable class
portable_code = '''"""
Portable U2Net Background Remover
Works on any system - downloads model if not present
Memory-optimized for web applications
"""
import torch
import numpy as np
import cv2
from torchvision import transforms
from PIL import Image
import sys
import os
import gc
import urllib.request
import zipfile
import shutil
import warnings
warnings.filterwarnings('ignore')

# Memory settings
torch.set_num_threads(1)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

class BackgroundRemover:
    """
    Portable U2Net Background Remover
    Automatically downloads U2Net if not found
    Memory-optimized for web applications
    """
    
    def __init__(self, model_path=None):
        self.device = torch.device('cpu')
        
        # Get paths
        self.current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
        self.project_root = os.path.dirname(self.current_dir) if '__file__' in dir() else self.current_dir
        
        # Set up paths
        self.u2net_dir = os.path.join(self.project_root, 'u2net')
        self.model_dir = os.path.join(self.u2net_dir, 'model')
        self.weights_dir = os.path.join(self.u2net_dir, 'weights')
        
        # Create directories
        os.makedirs(self.model_dir, exist_ok=True)
        os.makedirs(self.weights_dir, exist_ok=True)
        
        # Download/verify U2Net source
        self.u2net_available = self._ensure_u2net_source()
        
        # Set model path
        if model_path is None:
            model_path = os.path.join(self.weights_dir, 'u2net.pth')
        self.model_path = model_path
        
        # Import and load model
        self._import_u2net()
        self._load_model()
        
        # Training transform
        self.transform = transforms.Compose([
            transforms.Resize((320, 320)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
        ])
        
        gc.collect()
        if self.model is not None:
            print("✅ BackgroundRemover initialized with U2Net model")
        else:
            print("✅ BackgroundRemover initialized in fallback mode")
    
    def _ensure_u2net_source(self):
        """Download U2Net source if not present"""
        u2net_py = os.path.join(self.model_dir, 'u2net.py')
        
        if not os.path.exists(u2net_py):
            print("📥 Downloading U2Net source...")
            u2net_url = "https://github.com/xuebinqin/U-2-Net/archive/master.zip"
            zip_path = os.path.join(self.u2net_dir, 'u2net.zip')
            
            try:
                # Download
                urllib.request.urlretrieve(u2net_url, zip_path)
                
                # Extract
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(self.u2net_dir)
                
                # Move model files
                extracted = os.path.join(self.u2net_dir, 'U-2-Net-master')
                if os.path.exists(extracted):
                    src_model = os.path.join(extracted, 'model')
                    if os.path.exists(src_model):
                        for file in os.listdir(src_model):
                            if file.endswith('.py'):
                                shutil.copy(
                                    os.path.join(src_model, file),
                                    os.path.join(self.model_dir, file)
                                )
                
                # Clean up
                os.remove(zip_path)
                if os.path.exists(extracted):
                    shutil.rmtree(extracted)
                
                print("✅ U2Net source downloaded successfully")
                return True
                
            except Exception as e:
                print(f"⚠️ Could not download U2Net: {e}")
                return False
        else:
            print("✅ U2Net source already present")
            return True
    
    def _import_u2net(self):
        """Import U2Net module"""
        if self.model_dir not in sys.path:
            sys.path.insert(0, self.model_dir)
        
        try:
            from u2net import U2NET
            self.U2NET = U2NET
            self.u2net_available = True
        except ImportError:
            self.U2NET = None
            self.u2net_available = False
    
    def _load_model(self):
        """Load the U2Net model"""
        if self.U2NET is None:
            self.model = None
            return
            
        if not os.path.exists(self.model_path):
            self.model = None
            return
        
        try:
            self.model = self.U2NET(in_ch=3, out_ch=1)
            state_dict = torch.load(
                self.model_path, 
                map_location=self.device,
                weights_only=True
            )
            self.model.load_state_dict(state_dict)
            self.model.to(self.device)
            self.model.eval()
        except Exception:
            self.model = None
    
    def remove_background(self, image, target_size=(224, 224), max_size=800):
        """
        Memory-optimized background removal - preserves original leaf colors
        
        Args:
            image: numpy array or PIL Image
            target_size: final size for model input
            max_size: maximum dimension for processing
            
        Returns:
            result: numpy array with background removed (original colors preserved)
            mask: binary mask from U2Net
        """
        # Fallback if model not loaded
        if self.model is None:
            if isinstance(image, np.ndarray):
                return cv2.resize(image, target_size), None
            elif isinstance(image, Image.Image):
                return np.array(image.resize(target_size)), None
            return image, None
        
        try:
            # Store original image for color preservation
            original_image = image.copy() if isinstance(image, np.ndarray) else image
            
            # Resize large images for processing
            if isinstance(image, np.ndarray):
                h, w = image.shape[:2]
                if max(h, w) > max_size:
                    scale = max_size / max(h, w)
                    new_w = int(w * scale)
                    new_h = int(h * scale)
                    image = cv2.resize(image, (new_w, new_h))
                
                image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            elif isinstance(image, Image.Image):
                if max(image.size) > max_size:
                    scale = max_size / max(image.size)
                    new_size = tuple(int(dim * scale) for dim in image.size)
                    image = image.resize(new_size, Image.Resampling.LANCZOS)
                else:
                    image = image
            
            # Get mask
            input_tensor = self.transform(image).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                d1, d2, d3, d4, d5, d6, d7 = self.model(input_tensor)
            
            mask = d1[:, 0, :, :].cpu().numpy()
            
            # Clean up
            del d1, d2, d3, d4, d5, d6, d7, input_tensor
            gc.collect()
            
            # Process mask
            mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
            mask = mask.transpose(1, 2, 0)
            mask_target = cv2.resize(mask, target_size)
            mask_target = (mask_target * 255).astype(np.uint8)
            
            # === FIX: Preserve original colors ===
            # Get the original image resized to target size
            if isinstance(original_image, np.ndarray):
                image_resized = cv2.resize(original_image, target_size)
            else:
                image_resized = np.array(original_image.resize(target_size, Image.Resampling.LANCZOS))
            
            # Create result by copying the resized original
            result = image_resized.copy()
            
            # Handle mask dimensions
            if len(mask_target.shape) == 3 and mask_target.shape[2] == 1:
                mask_target = mask_target.squeeze()
            
            # Set background to white
            result[mask_target < 128] = [255, 255, 255]
            
            return result, mask_target
            
        except Exception as e:
            print(f"Warning: Background removal failed - {e}")
            if isinstance(image, Image.Image):
                return np.array(image.resize(target_size)), None
            return cv2.resize(image, target_size), None
    
    def __del__(self):
        try:
            gc.collect()
        except:
            pass
'''

# Save to utils folder
utils_dir = os.path.join(current_dir, 'utils')
os.makedirs(utils_dir, exist_ok=True)

with open(os.path.join(utils_dir, 'background_removal.py'), 'w', encoding='utf-8') as f:
    f.write(portable_code)
print(f"\n✅ Saved portable BackgroundRemover to {os.path.join(utils_dir, 'background_removal.py')}")

# %% [markdown]
# ### Create setup script for deployment

# %%
setup_code = '''"""
Setup script for U2Net
Run this once before deploying to download U2Net
"""
import os
import urllib.request
import zipfile
import shutil

def setup_u2net():
    """Download and setup U2Net"""
    
    # Create directories
    os.makedirs('u2net/model', exist_ok=True)
    os.makedirs('u2net/weights', exist_ok=True)
    
    # Download U2Net source
    print("📥 Downloading U2Net source...")
    u2net_url = "https://github.com/xuebinqin/U-2-Net/archive/master.zip"
    zip_path = "u2net/u2net_source.zip"
    
    try:
        urllib.request.urlretrieve(u2net_url, zip_path)
        
        # Extract
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("u2net/")
        
        # Copy model files
        source_model = os.path.join("u2net", "U-2-Net-master", "model")
        if os.path.exists(source_model):
            for file in os.listdir(source_model):
                if file.endswith('.py'):
                    shutil.copy(
                        os.path.join(source_model, file),
                        os.path.join("u2net", "model", file)
                    )
        
        # Clean up
        os.remove(zip_path)
        shutil.rmtree(os.path.join("u2net", "U-2-Net-master"))
        
        print("\n✅ U2Net source setup complete!")
        print("\n📌 Next step:")
        print("1. Download u2net.pth weights from:")
        print("   https://drive.google.com/uc?id=1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ")
        print("2. Place it in: u2net/weights/u2net.pth")
        
    except Exception as e:
        print(f"❌ Setup failed: {e}")

if __name__ == "__main__":
    setup_u2net()
'''

with open(os.path.join(current_dir, 'setup_u2net.py'), 'w', encoding='utf-8') as f:
    f.write(setup_code)
print("✅ Created setup_u2net.py - run this once before deploying")

# %% [markdown]
# ### Create requirements.txt with all dependencies

# %%
requirements = '''torch>=1.9.0
torchvision>=0.10.0
numpy>=1.21.0
opencv-python>=4.5.0
Pillow>=8.0.0
scikit-learn>=0.24.0
matplotlib>=3.3.0
flask>=2.0.0
werkzeug>=2.0.0
gdown  # For downloading weights
'''

with open(os.path.join(current_dir, 'requirements.txt'), 'w', encoding='utf-8') as f:
    f.write(requirements)
print("✅ Created requirements.txt")

# %% [markdown]
# ### Final Verification

# %%
print("\n" + "="*70)
print("🎯 PORTABLE BACKGROUND REMOVER - FINAL CHECK")
print("="*70)

print("\n✅ Created files:")
print("  • utils/background_removal.py - Portable U2Net class")
print("  • setup_u2net.py - Run once to download U2Net source")
print("  • requirements.txt - All dependencies")

print("\n📁 Current folder structure:")
print("  NitroSense-AI/")
print("  ├── app.py")
print("  ├── requirements.txt")
print("  ├── setup_u2net.py")
print("  ├── u2net/")
print("  │   ├── model/")
print("  │   │   └── u2net.py (source code)")
print("  │   └── weights/")
print(f"  │       └── u2net.pth {'✅' if os.path.exists(weights_path) else '❌'} (weights)")
print("  └── utils/")
print("      └── background_removal.py")

if os.path.exists(weights_path):
    print("\n✅ U2Net weights are present! Background removal will work.")
    print(f"\n📁 Test results saved to: {os.path.join(current_dir, 'test_output_with_weights')}")
    print("   Check the following files:")
    print("   - 6538a45fe4f02_original.jpg (original resized image)")
    print("   - 6538a45fe4f02_mask.jpg (U2Net mask)")
    print("   - 6538a45fe4f02_bg_removed.jpg (final result)")
else:
    print("\n⚠️ U2Net weights are missing!")
    print("   Download from: https://drive.google.com/uc?id=1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ")
    print(f"   Save to: {weights_path}")

print("\n📋 Deployment steps:")
print("1. Run: python setup_u2net.py (downloads U2Net source)")
print("2. Ensure u2net.pth is in u2net/weights/")
print("3. Install: pip install -r requirements.txt")
print("4. Run: python app.py")

print("\n✅ Your app is now fully portable!")
print("   Works on any system - no hardcoded paths!")

Current directory: C:\Users\HP\NitroSense-AI
✅ Created U2Net folders
✅ U2Net weights found: C:\Users\HP\NitroSense-AI\u2net\weights\u2net.pth
   File size: 168.12 MB

🧪 TESTING PORTABLE BACKGROUND REMOVER WITH ACTUAL WEIGHTS

✅ Found weights at: C:\Users\HP\NitroSense-AI\u2net\weights\u2net.pth

🔄 Initializing BackgroundRemover...
Using device: cpu
✅ U2Net source already present
✅ U2Net module imported
📦 Loading U2Net model from: C:\Users\HP\NitroSense-AI\u2net\weights\u2net.pth
✅ U2Net model loaded successfully!
   Model type: U2NET
✅ BackgroundRemover initialized with U2Net model

📷 Testing with: 2888723a-fd6f-4471-b7e5-692727bd5692_668bc6dbe4b48.jpg
  Original shape: (1717, 3162, 3)
  Original dtype: uint8
  Original range: [0, 241]

🔄 Test 1: Basic background removal...
  🎨 Background removal - Original colors preserved
     Mask range: [0, 254]
     Background pixels: 38454/50176
  ✅ Result shape: (224, 224, 3)
  ✅ Result dtype: uint8
  ✅ Result range: [0, 255]
  ✅ Mask shape: (22